In [ ]:
# ==========================================================
# Celda 01 — Importación de paquetes
# ==========================================================
import sys
import os

import numpy as np

import matplotlib
matplotlib.rcParams['text.usetex'] = True
import matplotlib.pyplot as plt

from scipy.linalg import sqrtm
from scipy.linalg import inv

sys.path.insert(0, '../')
from codes.Quantum import *
from codes.CSPSA import *


In [ ]:
# ==========================================================
# Celda 02 — Definición de funciones auxiliares
# ==========================================================

def complex2unitary( Z ):
    if Z.ndim == 3:
        for k, z in enumerate(Z):
            Z[k] = z @ inv( sqrtm( z.T.conj()@z ) )
    else:
        Z = Z @ inv( sqrtm( Z.T.conj()@Z ) )
    return Z 

def random_unitary( dim, num=1 ):
    Z = np.random.randn( num, dim, dim ) \
        + 1j*np.random.randn( num, dim, dim )
    return complex2unitary( Z ).squeeze()

def postprocessing(U):
    # U, _ = np.linalg.qr(U)
    # # U = U@(U.T.conj()U).
    return complex2unitary( U )

def trineq(Us, Psi ):

    Us = np.array(Us)
    Us = complex2unitary( Us )

    # M_per_system = []
    # for U in Us:
    #     M_each_qubit = []
    #     for u in U:
    #         M_each_qubit.append( np.outer(u,u.conj()) )
    #     M_per_system.append( M_each_qubit )
    # probs = InnerProductMatrices( Rho, M_per_system ).real
    # probs = probs.reshape(6*[2])

    probs = np.abs( LocalProduct( Psi, Us ).reshape(6*[2]) )**2

    # Z  = np.array(Z)
    # Z = complex2unitary( Z )
    # Measures = triple_kron( Z[0], Z[1], Z[2] )
    # probs    = np.sum( Measures.conj() * ( Rho @ Measures ), 
    #                     axis=0 ).reshape(6*[2]).real

    probs_albl = np.sum( probs, axis=(1,3,4,5) )
    probs_alblclcr = np.sum( probs, axis=(1,3) )
    probs_clcr = np.sum( probs, axis=(0,1,2,3) )

    inequality = (-probs_albl[1,0] 
                +probs_alblclcr[1,0,1,0] 
                -probs_albl[0,1] * probs_clcr[1,0] 
                -probs_clcr[0,0] * probs_clcr[1,1] 
                -probs_clcr[0,1] * probs[1,0,0,1,1,0]
                -probs_clcr[0,1] * probs[1,1,0,0,1,0]
                +probs_clcr[0,0] * probs[1,0,1,1,1,1]
                +probs_clcr[0,0] * probs[1,1,1,0,1,1]
                +probs_clcr[1,0] * probs[0,0,1,0,0,1]
                +probs_clcr[1,0] * probs[0,1,1,1,0,1]
                +probs_clcr[1,1] * probs[0,0,0,0,0,0]
                +probs_clcr[1,1] * probs[0,1,0,1,0,0]
                )
    return inequality


In [ ]:
# ==========================================================
# Celda 03 — Mediciones óptimas y validación del funcional trineq
# ==========================================================
M = np.zeros( (2,2,2,2) )

M[0,1,:,:] = np.array([  [1, 0], [0, 1]  ]) #Z
M[0,0,:,:] = np.array([  [1, 1], [1, -1]  ])/np.sqrt(2) #X

M[1,0,0,:] = np.array([1-np.sqrt(2), 1])/np.sqrt( 2*(2-np.sqrt(2)) )  #X+Z
M[1,0,1,:] = np.array([1+np.sqrt(2), 1])/np.sqrt( 2*(2+np.sqrt(2)) )

M[1,1,0,:] = np.array([-1-np.sqrt(2), 1])/np.sqrt( 2*(2+np.sqrt(2)) )
M[1,1,1,:] = np.array([-1+np.sqrt(2), 1])/np.sqrt( 2*(2-np.sqrt(2)) ) #

Comp_basis = np.eye(2) 

vec_alice = np.zeros( (2,2,4) )
for l in range(2):
    for k in range(2):
        vec_alice[l,k,:]  = np.kron( Comp_basis[l], M[0,l,k,:] )
vec_alice = vec_alice.reshape(4,4)

vec_bob = np.zeros( (2,2,4) )
for m in range(2):
    for kk in range(2):
        vec_bob[m,kk,:] = np.kron( Comp_basis[m], M[1,m,kk,:] )
vec_bob = vec_bob.reshape(4,4)

Comp_basis = np.array([ 0,1,1,0 ]).reshape(2,2)
vec_charlie = np.zeros( (2,2,4) )
for i in range(2):
    for j in range(2):
        vec_charlie[i,j,:] = np.kron( Comp_basis[i], Comp_basis[j] )
vec_charlie = vec_charlie.reshape(4,4)

Psi_ab = np.array([ 0, 1, -1, 0 ])/np.sqrt(2)
Psi = triple_kron( Psi_ab, Psi_ab, Psi_ab )
Psi = Psi.reshape(6*[2]).transpose([5,0,2,1,4,3]).flatten() 
Rho = np.outer( Psi, Psi.conj() )

trineq( np.array([vec_alice, vec_bob, vec_charlie]), Psi ), (np.sqrt(2)-1)/16 


In [ ]:
# ==========================================================
# Celda 04 — Documentación: convención de índices por detector
# ==========================================================
# alpha=0,1 beta=2,3 gamma=4,5 
# alice  5,0 ( alpha, gamma ) 
# bob    2,1 ( beta, alpha )
# charli 4,3 ( gamma, beta )


In [ ]:
# ==========================================================
# Celda 05 — Verificación de la forma del estado Psi
# ==========================================================
Psi = triple_kron( Psi_ab, Psi_ab, Psi_ab )
Psi.shape ## 64 = 2**6 


In [ ]:
# ==========================================================
# Celda 06 — Documentación: convención de coeficientes de Psi
# ==========================================================
## |Psi> = sum_{ijklmn} Psi_{ijklmn} |i>|j>|k>|l>|m>|n>
#                                     2  2  2  2  2  2 


In [ ]:
# ==========================================================
# Celda 07 — Ejemplo de acceso a componentes de Psi
# ==========================================================
Psi.reshape([2,2,2,2,2,2])[0,1,1]
# i=0, j=1, k=1 ---> Psi_{011lmn}


In [ ]:
# ==========================================================
# Celda 08 — Simulación principal: barrido en alpha con CSPSA
# ==========================================================
gains = [3, 0, 0.1, 0.602, 0.101]
max_iter = 1000
max_rep = 100
Out = []
iter = np.linspace(0.0, 0.5, 6)

Out = np.empty((max_rep, max_iter))

for i in range(6):
    a = i/10

    ####
    psi = np.sqrt(a)*np.array([[0, 1, 0, 0]]) + np.sqrt(1-a)*np.array([[0, 0, 1, 0]])
    Psi = triple_kron( psi, psi, psi )
    Psi = Psi.reshape(6*[2]).transpose([5,0,2,1,4,3]).flatten() 
    ####

    print( psi )
    for k in range(max_rep):
        Us_in = np.array([ np.kron(random_unitary(2), random_unitary(2)) for _ in range(3) ]) 
        cspsa = CSPSA(gains=gains, minimize=False)
        
        results = cspsa.minimize(trineq, Us_in, 
                                    max_iter, 
                                    args = (Psi), 
                                    postprocessing=postprocessing )
        infidelities = results.fun

        Out[k, :] = infidelities
    
    np.save("Data_sim_0"+str(i), Out)
    #1m6s 


In [ ]:
# ==========================================================
# Celda 09 — Carga de datos de una simulación individual
# ==========================================================
Load = "Data_sim_00.npy"
Data = np.load(Load)


In [ ]:
# ==========================================================
# Celda 10 — Graficación de curvas de convergencia por alpha
# ==========================================================
plt.figure()
max_iter = Data.shape[1]

n = 6
colors = plt.cm.jet(np.linspace(0,1,n))
for i in range(6):
    Load = "Data_sim_0"+str(i)+".npy"
    Data = np.load(Load)

    #Data_mean = np.mean(data, axis=0)
    Data_median = np.median(Data, axis=0)
    Label = r'$\alpha = 0.$'+rf'${i}$'
    plt.plot(Data_median, color = colors[i], label = Label)
#plt.hlines( (np.sqrt(2)-1)/16 , 0, max_iter, linestyles='--' )


nuevo_tick = 0.02588

# Obtener el eje actual
ax = plt.gca()

# Obtener ticks actuales del eje Y
yticks = ax.get_yticks()

# Agregar el nuevo tick
yticks = np.sort(np.append(yticks, nuevo_tick))

# Fijar los ticks
ax.set_yticks(yticks)
#plt.hlines(0, 0, 1000, linestyles='--')
plt.xlabel(r"Iterations")
plt.ylabel(r"Inequality value")

plt.legend()
plt.grid()
plt.savefig("t0"+str(i)+".png")
plt.savefig("schmidt.pdf")
